In [1]:
import datetime as dt
import pandas as pd
import numpy as np
import yfinance as yf
import plotly.io as pio
import plotly.graph_objects as go
import scipy.stats as stats
from plotly.subplots import make_subplots
import plotly.offline as pyo
pyo.init_notebook_mode(connected = True)
pio.renderers.default = 'iframe'
pd.options.plotting.backend = 'plotly'
import pylab


In [2]:
start = dt.datetime(2025,5,7)
end = dt.datetime.now()
start,end

(datetime.datetime(2025, 5, 7, 0, 0),
 datetime.datetime(2026, 5, 7, 23, 37, 18, 278964))

In [3]:
stocks = ['^GSPC','AMD', 'INTC', 'NVDA', 'AVGO', 'TXN', 'TSM','LRCX']

In [4]:
df = yf.download(stocks,start,end)
log_returns = np.log(df.Close/df.Close.shift(1)).dropna()
log_returns.head()

[*********************100%***********************]  8 of 8 completed


Ticker,AMD,AVGO,INTC,LRCX,NVDA,TSM,TXN,^GSPC
Date,,,,,,,,
2025-05-08,0.013264,0.014349,0.033409,-0.002134,0.002645,0.003888,0.005145,0.005783
2025-05-09,0.011147,0.002067,0.019803,0.006123,-0.006153,0.007392,0.039246,-0.000712
2025-05-12,0.050067,0.062285,0.034866,0.087487,0.053006,0.057567,0.083542,0.032040
2025-05-13,0.039356,0.047762,0.016987,0.035825,0.054811,0.036805,0.004900,0.007222
2025-05-14,0.045711,-0.001292,-0.047196,-0.003290,0.040794,0.003961,-0.004580,0.001024


In [5]:
def calc_beta(df):
    np_array = df.values
    #market index m 
    m = np_array[:,7]
    beta = []
    for ind, col in enumerate(df):
        if ind < 7:
            s = np_array[:,ind]
            covariance = np.cov(s,m)
            beta.append(covariance[0,1]/covariance[1,1])
    return pd.Series(beta, df.columns[:7], name = "Beta")

In [6]:
calc_beta(log_returns)

Ticker
AMD     2.507625
AVGO    1.954299
INTC    2.183888
LRCX    2.616604
NVDA    1.748255
TSM     1.906621
TXN     1.095098
Name: Beta, dtype: float64

In [7]:
def regression_beta(df):
    np_array = df.values
    #market index m 
    m = np_array[:,7]
    beta = []
    for ind, col in enumerate(df):
        if ind < 7:
            s = np_array[:,ind]
            beta.append(stats.linregress(m,s)[0])
    return pd.Series(beta, df.columns[:7], name = "Beta")

In [8]:
regression_beta(log_returns)

Ticker
AMD     2.507625
AVGO    1.954299
INTC    2.183888
LRCX    2.616604
NVDA    1.748255
TSM     1.906621
TXN     1.095098
Name: Beta, dtype: float64

In [9]:
def martix_beta(df):
    x = df.values[:,[7]]
    x = np.concatenate([np.ones_like(x),x], axis = 1)
    beta = np.linalg.pinv(x.T @ x) @ x.T @ df.values[:, :7]
    return pd.Series(beta[1],df.columns[:7], name = "Beta")

In [10]:
beta = martix_beta(log_returns)

In [11]:
units = np.array([100,120,1000,300,200,140, 150])
snpPrices = df.Close[-1:].values.tolist()[0]
price = np.array([round(price,2) for price in snpPrices[:7]])
value = units*price
weight = [round(val/sum(value),2) for val in value]
beta = round(beta,2)

In [12]:
portfolio = pd.DataFrame({
    'Stock' : stocks[1:],
    'Direction' : 'Long',
    'Type':'S',
    'Stock Price': price,
    'Price' : price, 
    'Units' : units, 
    'Value' : value, 
    'Weights' : weight,
    'Beta' : beta,
    'Weighted Beta' : weight*beta
})
portfolio

,Stock,Direction,Type,Stock Price,Price,Units,Value,Weights,Beta,Weighted Beta
Ticker,,,,,,,,,,
AMD,AMD,Long,S,408.50,408.50,100,40850.0,0.09,2.51,0.2259
AVGO,INTC,Long,S,415.46,415.46,120,49855.2,0.12,1.95,0.2340
INTC,NVDA,Long,S,111.04,111.04,1000,111040.0,0.26,2.18,0.5668
LRCX,AVGO,Long,S,287.04,287.04,300,86112.0,0.20,2.62,0.5240
NVDA,TXN,Long,S,213.06,213.06,200,42612.0,0.10,1.75,0.1750
TSM,TSM,Long,S,412.72,412.72,140,57780.8,0.13,1.91,0.2483
TXN,LRCX,Long,S,284.32,284.32,150,42648.0,0.10,1.10,0.1100


In [13]:
portfolio = portfolio.drop(['Weighted Beta', 'Weights'], axis = 1)
portfolio['Delta'] = portfolio['Units']
portfolio

,Stock,Direction,Type,Stock Price,Price,Units,Value,Beta,Delta
Ticker,,,,,,,,,
AMD,AMD,Long,S,408.50,408.50,100,40850.0,2.51,100
AVGO,INTC,Long,S,415.46,415.46,120,49855.2,1.95,120
INTC,NVDA,Long,S,111.04,111.04,1000,111040.0,2.18,1000
LRCX,AVGO,Long,S,287.04,287.04,300,86112.0,2.62,300
NVDA,TXN,Long,S,213.06,213.06,200,42612.0,1.75,200
TSM,TSM,Long,S,412.72,412.72,140,57780.8,1.91,140
TXN,LRCX,Long,S,284.32,284.32,150,42648.0,1.10,150


In [14]:
Options = [
    {'option':'AMD_C_405_Jun05', 'underlying':'AMD',  'price': 30.00, 'units': 2, 'delta':  0.552, 'direction': 'Short', 'type': 'Call'},
    {'option':'NVDA_P_212_Jun18','underlying':'NVDA', 'price': 12.01, 'units': 2, 'delta': -0.453, 'direction': 'Long',  'type': 'Put'}
]

In [17]:
for row in Options:
    portfolio.loc[row['option']] = [
        row['underlying'],
        row['direction'],
        row['type'],
        portfolio.loc[row['underlying'], 'Stock Price'],
        row['price'],
        row['units'],
        row['price'] * row['units'] * 100,
        beta[row['underlying']],
        row['delta'] * row['units'] * 100 if row['direction'] == 'Long' else -row['delta'] * row['units'] * 100
    ]
portfolio

,Stock,Direction,Type,Stock Price,Price,Units,Value,Beta,Delta
Ticker,,,,,,,,,
AMD,AMD,Long,S,408.50,408.50,100,40850.0,2.51,100.0
AVGO,INTC,Long,S,415.46,415.46,120,49855.2,1.95,120.0
INTC,NVDA,Long,S,111.04,111.04,1000,111040.0,2.18,1000.0
LRCX,AVGO,Long,S,287.04,287.04,300,86112.0,2.62,300.0
NVDA,TXN,Long,S,213.06,213.06,200,42612.0,1.75,200.0
TSM,TSM,Long,S,412.72,412.72,140,57780.8,1.91,140.0
TXN,LRCX,Long,S,284.32,284.32,150,42648.0,1.10,150.0
AMD_C_405_Jun05,AMD,Short,Call,408.50,30.00,2,6000.0,2.51,-110.4
NVDA_P_212_Jun18,NVDA,Long,Put,213.06,12.01,2,2402.0,1.75,-90.6


In [18]:
portfolio['snp500 weighted delta (point)'] = round(portfolio['Beta']*portfolio['Stock Price']/snpPrices[0] * portfolio['Delta'],2)
portfolio['snp500 weighted delta (1%)'] = round(portfolio['Beta']*portfolio['Stock Price'] * portfolio['Delta'] * 0.01,2)
portfolio

,Stock,Direction,Type,Stock Price,Price,Units,Value,Beta,Delta,snp500 weighted delta (point),snp500 weighted delta (1%)
Ticker,,,,,,,,,,,
AMD,AMD,Long,S,408.50,408.50,100,40850.0,2.51,100.0,251.00,1025.33
AVGO,INTC,Long,S,415.46,415.46,120,49855.2,1.95,120.0,237.99,972.18
INTC,NVDA,Long,S,111.04,111.04,1000,111040.0,2.18,1000.0,592.58,2420.67
LRCX,AVGO,Long,S,287.04,287.04,300,86112.0,2.62,300.0,552.30,2256.13
NVDA,TXN,Long,S,213.06,213.06,200,42612.0,1.75,200.0,182.55,745.71
TSM,TSM,Long,S,412.72,412.72,140,57780.8,1.91,140.0,270.16,1103.61
TXN,LRCX,Long,S,284.32,284.32,150,42648.0,1.10,150.0,114.84,469.13
AMD_C_405_Jun05,AMD,Short,Call,408.50,30.00,2,6000.0,2.51,-110.4,-277.10,-1131.97
NVDA_P_212_Jun18,NVDA,Long,Put,213.06,12.01,2,2402.0,1.75,-90.6,-82.69,-337.81


In [20]:
portfolio.loc['Total', ['Value', 'snp500 weighted delta (point)', 'snp500 weighted delta (1%)']] = portfolio[['Value', 'snp500 weighted delta (point)', 'snp500 weighted delta (1%)']].sum()
portfolio

,Stock,Direction,Type,Stock Price,Price,Units,Value,Beta,Delta,snp500 weighted delta (point),snp500 weighted delta (1%)
Ticker,,,,,,,,,,,
AMD,AMD,Long,S,408.50,408.50,100.0,40850.0,2.51,100.0,251.00,1025.33
AVGO,INTC,Long,S,415.46,415.46,120.0,49855.2,1.95,120.0,237.99,972.18
INTC,NVDA,Long,S,111.04,111.04,1000.0,111040.0,2.18,1000.0,592.58,2420.67
LRCX,AVGO,Long,S,287.04,287.04,300.0,86112.0,2.62,300.0,552.30,2256.13
NVDA,TXN,Long,S,213.06,213.06,200.0,42612.0,1.75,200.0,182.55,745.71
TSM,TSM,Long,S,412.72,412.72,140.0,57780.8,1.91,140.0,270.16,1103.61
TXN,LRCX,Long,S,284.32,284.32,150.0,42648.0,1.10,150.0,114.84,469.13
AMD_C_405_Jun05,AMD,Short,Call,408.50,30.00,2.0,6000.0,2.51,-110.4,-277.10,-1131.97
NVDA_P_212_Jun18,NVDA,Long,Put,213.06,12.01,2.0,2402.0,1.75,-90.6,-82.69,-337.81
